# Нормализација на Yelp податоците за PostgreSQL

Во овој дел е прикажана подготовката на веќе исчистените Yelp CSV датотеки од `processed/` за внесување во PostgreSQL. Наместо повторно да се читаат оригиналните JSON Lines датотеки, notebook-от ги користи processed CSV датотеките и од нив ги создава нормализираните CSV датотеки погодни за `COPY` внесување.

Понатаму се дефинира релациски модел и партиции што ќе се користат за споредба со MongoDB. Индексите ќе се додадат подоцна, откако ќе се финализираат прашалниците за мерење.


## 1. Конфигурација

In [1]:
from pathlib import Path
import ast
import csv
import json
import sys
from datetime import datetime
from collections import Counter

while True:
    try:
        csv.field_size_limit(sys.maxsize)
        break
    except OverflowError:
        sys.maxsize //= 10

# Supports running the notebook either from the repo root or from the postgres/ folder.
DATA_DIR_CANDIDATES = [Path("postgres") / "processed", Path("processed")]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), DATA_DIR_CANDIDATES[0])
OUTPUT_DIR = Path("postgres_output")
CSV_DIR = OUTPUT_DIR / "csv"
SQL_DIR = OUTPUT_DIR / "sql"

CSV_DIR.mkdir(parents=True, exist_ok=True)
SQL_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "business": DATA_DIR / "businesses.csv",
    "user": DATA_DIR / "users.csv",
    "review": DATA_DIR / "reviews.csv",
    "tip": DATA_DIR / "tips.csv",
    "checkin": DATA_DIR / "checkins.csv",
}

for name, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Недостасува processed датотека за {name}: {path}. "
            "Провери дали processed CSV датотеките се во postgres/processed/ или processed/."
        )

print("Input processed CSV:", DATA_DIR.resolve())
print("Output CSV:", CSV_DIR.resolve())
print("Output SQL:", SQL_DIR.resolve())


Input processed CSV: /mnt/c/Users/RazorVision/Desktop/NBP_Proekt/postgres/processed
Output CSV: /mnt/c/Users/RazorVision/Desktop/NBP_Proekt/postgres/postgres_output/csv
Output SQL: /mnt/c/Users/RazorVision/Desktop/NBP_Proekt/postgres/postgres_output/sql


## 2. Чистење и нормализација на податоците

Пред внесување во PostgreSQL се прави финална подготовка и проверка на веќе исчистените CSV податоци:

- празни стрингови, `Unknown`, `None` и `NaN` се претвораат во `NULL`
- датумите се проверуваат и стандардизираат како `YYYY-MM-DD HH:MM:SS`
- полињата со повеќе вредности, како категории и пријатели, се претвораат во повеќе редови
- `attributes` и `hours`, кои во processed CSV се зачувани како serialized Python dict/list вредности, се издвојуваат во посебни табели
- checkin листата од processed CSV се претвора во еден ред по timestamp
- редови без задолжителни клучеви се отфрлаат
- редови што референцираат непостоечки `business_id` или `user_id` се отфрлаат за да се зачува референцијалниот интегритет

In [2]:
DATETIME_FORMAT = "%Y-%m-%d %H:%M:%S"
MISSING_VALUES = {"", "unknown", "none", "nan", "null"}
COMPLIMENT_FIELDS = [
    "compliment_hot",
    "compliment_more",
    "compliment_profile",
    "compliment_cute",
    "compliment_list",
    "compliment_note",
    "compliment_plain",
    "compliment_cool",
    "compliment_funny",
    "compliment_writer",
    "compliment_photos",
]
HOUR_COLUMNS = {
    "hours_Monday": "Monday",
    "hours_Tuesday": "Tuesday",
    "hours_Wednesday": "Wednesday",
    "hours_Thursday": "Thursday",
    "hours_Friday": "Friday",
    "hours_Saturday": "Saturday",
    "hours_Sunday": "Sunday",
}

def is_missing(value):
    if value is None:
        return True
    return str(value).strip().lower() in MISSING_VALUES

def clean_text(value):
    if is_missing(value):
        return None
    value = str(value).strip()
    return value if value else None

def clean_int(value):
    if is_missing(value):
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        try:
            return int(float(value))
        except (TypeError, ValueError):
            return None

def clean_float(value):
    if is_missing(value):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def clean_bool(value):
    if is_missing(value):
        return None
    if isinstance(value, bool):
        return value
    if isinstance(value, int):
        return value == 1
    value = str(value).strip().lower()
    if value in {"1", "true", "yes"}:
        return True
    if value in {"0", "false", "no"}:
        return False
    return None

def clean_datetime(value):
    value = clean_text(value)
    if value is None:
        return None
    try:
        return datetime.strptime(value, DATETIME_FORMAT).strftime(DATETIME_FORMAT)
    except ValueError:
        return None

def normalize_time(value):
    value = clean_text(value)
    if value is None or ":" not in value:
        return None
    try:
        hour, minute = value.split(":", 1)
        return f"{int(hour):02d}:{int(minute):02d}:00"
    except ValueError:
        return None

def split_hours(value):
    value = clean_text(value)
    if value is None or "-" not in value:
        return None, None
    open_raw, close_raw = value.split("-", 1)
    return normalize_time(open_raw), normalize_time(close_raw)

def parse_serialized(value, expected_type):
    if isinstance(value, expected_type):
        return value
    value = clean_text(value)
    if value is None:
        return expected_type()
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return expected_type()
    return parsed if isinstance(parsed, expected_type) else expected_type()

def parse_list(value):
    if isinstance(value, list):
        return value
    value = clean_text(value)
    if value is None:
        return []
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return [part.strip() for part in value.split(",") if part.strip()]
    return parsed if isinstance(parsed, list) else []

def parse_dict(value):
    return parse_serialized(value, dict)

def normalize_attribute_value(value):
    if is_missing(value):
        return None

    if isinstance(value, str):
        value = value.strip()
        if not value:
            return None

        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return value

    return value


def serialize_attribute_value(value):
    normalized = normalize_attribute_value(value)
    if normalized is None:
        return None

    return json.dumps(normalized, ensure_ascii=False, sort_keys=True)

def write_dict_csv(path, fieldnames):
    file_obj = path.open("w", newline="", encoding="utf-8")
    writer = csv.DictWriter(file_obj, fieldnames=fieldnames)
    writer.writeheader()
    return file_obj, writer

def iter_csv(path):
    with path.open("r", newline="", encoding="utf-8") as f:
        yield from csv.DictReader(f)


## 3. Мал пример пред целосната трансформација

Во овој дел прво се прикажува еден `business` запис пред нормализација, а потоа се прикажуваат релациските записи што се добиваат со примена на трансформациските функции.

In [3]:
sample_business = next(iter_csv(FILES["business"]))
sample_business


{'business_id': 'Pns2l4eNsfO8kk83dixA6A',
 'name': 'Abby Rappoport, LAC, CMQ',
 'address': '1616 Chapala St, Ste 2',
 'city': 'Santa Barbara',
 'state': 'CA',
 'postal_code': '93101',
 'latitude': '34.4266787',
 'longitude': '-119.7111968',
 'stars': '5.0',
 'review_count': '7',
 'is_open': '0',
 'attributes': "{'ByAppointmentOnly': 'True'}",
 'categories': "['Doctors', 'Traditional Chinese Medicine', 'Naturopathic/Holistic', 'Acupuncture', 'Health & Medical', 'Nutritionists']",
 'hours': 'Unknown',
 'ByAppointmentOnly': 'True',
 'BusinessAcceptsCreditCards': 'Unknown',
 'BikeParking': 'Unknown',
 'RestaurantsPriceRange2': 'Unknown',
 'CoatCheck': 'Unknown',
 'RestaurantsTakeOut': 'Unknown',
 'RestaurantsDelivery': 'Unknown',
 'Caters': 'Unknown',
 'WiFi': 'Unknown',
 'BusinessParking': 'Unknown',
 'WheelchairAccessible': 'Unknown',
 'HappyHour': 'Unknown',
 'OutdoorSeating': 'Unknown',
 'HasTV': 'Unknown',
 'RestaurantsReservations': 'Unknown',
 'DogsAllowed': 'Unknown',
 'Alcohol': '

In [4]:
def transform_business(row):
    return {
        "business_id": clean_text(row.get("business_id")),
        "name": clean_text(row.get("name")),
        "address": clean_text(row.get("address")),
        "city": clean_text(row.get("city")),
        "state": clean_text(row.get("state")),
        "postal_code": clean_text(row.get("postal_code")),
        "latitude": clean_float(row.get("latitude")),
        "longitude": clean_float(row.get("longitude")),
        "stars": clean_float(row.get("stars")),
        "review_count": clean_int(row.get("review_count")),
        "is_open": clean_bool(row.get("is_open")),
    }

def extract_business_categories(row):
    business_id = clean_text(row.get("business_id"))
    categories = parse_list(row.get("categories"))
    if business_id is None:
        return []
    return [
        {"business_id": business_id, "category_name": category}
        for category in (clean_text(category) for category in categories)
        if category is not None
    ]

def extract_business_attributes(row):
    business_id = clean_text(row.get("business_id"))
    attributes = parse_dict(row.get("attributes"))
    if business_id is None:
        return []
    return [
        {
            "business_id": business_id,
            "attribute_name": clean_text(name),
            "attribute_value": serialize_attribute_value(value),
        }
        for name, value in attributes.items()
        if clean_text(name) is not None
    ]

def extract_business_hours(row):
    business_id = clean_text(row.get("business_id"))
    hours = parse_dict(row.get("hours"))

    if not hours:
        hours = {
            day: row.get(column)
            for column, day in HOUR_COLUMNS.items()
            if clean_text(row.get(column)) is not None
        }

    if business_id is None:
        return []

    result = []
    for day, value in hours.items():
        open_time, close_time = split_hours(value)
        if clean_text(day) is None or (open_time is None and close_time is None):
            continue
        result.append({
            "business_id": business_id,
            "day_of_week": clean_text(day),
            "open_time": open_time,
            "close_time": close_time,
        })
    return result

print("businesses row:")
print(transform_business(sample_business))
print()
print("categories rows:")
print(extract_business_categories(sample_business))
print()
print("attributes rows:")
print(extract_business_attributes(sample_business))
print()
print("hours rows:")
print(extract_business_hours(sample_business))


businesses row:
{'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'name': 'Abby Rappoport, LAC, CMQ', 'address': '1616 Chapala St, Ste 2', 'city': 'Santa Barbara', 'state': 'CA', 'postal_code': '93101', 'latitude': 34.4266787, 'longitude': -119.7111968, 'stars': 5.0, 'review_count': 7, 'is_open': False}

categories rows:
[{'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_name': 'Doctors'}, {'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_name': 'Traditional Chinese Medicine'}, {'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_name': 'Naturopathic/Holistic'}, {'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_name': 'Acupuncture'}, {'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_name': 'Health & Medical'}, {'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_name': 'Nutritionists'}]

attributes rows:
[{'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'attribute_name': 'ByAppointmentOnly', 'attribute_value': 'true'}]

hours rows:
[]


## 4. Трансформација на businesses, categories, attributes и hours

Во овој дел business податоците се разделуваат во неколку релациски табели: `businesses`, `categories`, `business_categories`, `business_attributes` и `business_hours`.

Основните business информации се запишуваат во `businesses`, категориите се издвојуваат во many-to-many модел, attributes се чуваат како `JSONB`, а работното време се запишува како посебни редови по ден.

In [5]:
business_fields = ["business_id", "name", "address", "city", "state", "postal_code", "latitude", "longitude", "stars", "review_count", "is_open"]
category_fields = ["category_name"]
business_category_fields = ["business_id", "category_name"]
business_attribute_fields = ["business_id", "attribute_name", "attribute_value"]
business_hour_fields = ["business_id", "day_of_week", "open_time", "close_time"]

valid_business_ids = set()
category_names = set()
business_category_pairs = set()
business_counts = Counter()

files_to_close = []
try:
    businesses_file, businesses_writer = write_dict_csv(CSV_DIR / "businesses.csv", business_fields)
    business_categories_file, business_categories_writer = write_dict_csv(CSV_DIR / "business_categories.csv", business_category_fields)
    business_attributes_file, business_attributes_writer = write_dict_csv(CSV_DIR / "business_attributes.csv", business_attribute_fields)
    business_hours_file, business_hours_writer = write_dict_csv(CSV_DIR / "business_hours.csv", business_hour_fields)
    files_to_close.extend([businesses_file, business_categories_file, business_attributes_file, business_hours_file])

    for raw in iter_csv(FILES["business"]):
        row = transform_business(raw)
        business_id = row["business_id"]

        if business_id is None or row["name"] is None:
            business_counts["dropped_missing_required"] += 1
            continue
        valid_business_ids.add(business_id)
        businesses_writer.writerow(row)
        business_counts["businesses"] += 1

        for category_row in extract_business_categories(raw):
            category_name = category_row["category_name"]
            category_names.add(category_name)
            pair = (category_row["business_id"], category_name)
            if pair not in business_category_pairs:
                business_category_pairs.add(pair)
                business_categories_writer.writerow(category_row)
                business_counts["business_categories"] += 1

        for attribute_row in extract_business_attributes(raw):
            business_attributes_writer.writerow(attribute_row)
            business_counts["business_attributes"] += 1

        for hour_row in extract_business_hours(raw):
            business_hours_writer.writerow(hour_row)
            business_counts["business_hours"] += 1
finally:
    for f in files_to_close:
        f.close()

with (CSV_DIR / "categories.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=category_fields)
    writer.writeheader()
    for category_name in sorted(category_names):
        writer.writerow({"category_name": category_name})
        business_counts["categories"] += 1

business_counts


Counter({'business_attributes': 1206820,
         'business_hours': 801015,
         'business_categories': 668549,
         'businesses': 150346,
         'categories': 1311})

## 5. Контролна листа на валидни users

Во овој дел се создава множество од валидни `user_id` вредности.

Ова множество подоцна се користи за проверка дали reviews, tips и friendships референцираат постоечки корисници.

In [6]:
def collect_valid_user_ids():
    user_ids = set()
    counts = Counter()
    for raw in iter_csv(FILES["user"]):
        user_id = clean_text(raw.get("user_id"))
        if user_id is not None:
            user_ids.add(user_id)
            counts["users"] += 1
    return user_ids, counts

valid_user_ids, user_id_counts = collect_valid_user_ids()

print("valid businesses:", len(valid_business_ids))
print("valid users:", len(valid_user_ids))
user_id_counts


valid businesses: 150346
valid users: 1000000


Counter({'users': 1000000})

## 6. Трансформација на users, compliments и friends

Во овој дел user податоците се разделуваат во `users`, `user_compliments` и `user_friends`.

Главните информации за корисниците се запишуваат во `users`, а листата на elite години се претвора во summary полиња: `elite_years_count`, `elite_first_year`, `elite_last_year` и `is_elite`. Compliment вредностите се претвораат во посебни редови, а листата на friends се претвора во врски меѓу корисници.

In [7]:
def transform_user(row):
    return {
        "user_id": clean_text(row.get("user_id")),
        "name": clean_text(row.get("name")),
        "review_count": clean_int(row.get("review_count")),
        "yelping_since": clean_datetime(row.get("yelping_since")),
        "useful": clean_int(row.get("useful")),
        "funny": clean_int(row.get("funny")),
        "cool": clean_int(row.get("cool")),
        "fans": clean_int(row.get("fans")),
        "average_stars": clean_float(row.get("average_stars")),
        "elite_years_count": clean_int(row.get("elite_years_count")),
        "elite_first_year": clean_int(row.get("elite_first_year")),
        "elite_last_year": clean_int(row.get("elite_last_year")),
        "is_elite": clean_bool(row.get("is_elite")),
    }

def extract_user_compliments(row):
    user_id = clean_text(row.get("user_id"))
    if user_id is None:
        return []
    result = []
    for field in COMPLIMENT_FIELDS:
        count = clean_int(row.get(field)) or 0
        if count > 0:
            result.append({
                "user_id": user_id,
                "compliment_type": field.replace("compliment_", ""),
                "compliment_count": count,
            })
    return result

def extract_user_friends(row):
    user_id = clean_text(row.get("user_id"))
    friends = clean_text(row.get("friends"))
    if user_id is None or friends is None:
        return []
    result = []
    seen = set()
    for friend_id in friends.split(","):
        friend_id = clean_text(friend_id)
        if friend_id is None or friend_id == user_id or friend_id not in valid_user_ids:
            continue
        pair = (user_id, friend_id)
        if pair not in seen:
            seen.add(pair)
            result.append({"user_id": user_id, "friend_user_id": friend_id})
    return result

user_fields = [
    "user_id", "name", "review_count", "yelping_since", "useful", "funny", "cool", "fans", "average_stars",
    "elite_years_count", "elite_first_year", "elite_last_year", "is_elite",
]
compliment_fields = ["user_id", "compliment_type", "compliment_count"]
friend_fields = ["user_id", "friend_user_id"]

user_counts = Counter()
files_to_close = []
try:
    users_file, users_writer = write_dict_csv(CSV_DIR / "users.csv", user_fields)
    compliments_file, compliments_writer = write_dict_csv(CSV_DIR / "user_compliments.csv", compliment_fields)
    friends_file, friends_writer = write_dict_csv(CSV_DIR / "user_friends.csv", friend_fields)
    files_to_close.extend([users_file, compliments_file, friends_file])

    for raw in iter_csv(FILES["user"]):
        row = transform_user(raw)
        user_id = row["user_id"]

        if user_id is None:
            user_counts["dropped_missing_user_id"] += 1
            continue
        users_writer.writerow(row)
        user_counts["users"] += 1

        for compliment_row in extract_user_compliments(raw):
            compliments_writer.writerow(compliment_row)
            user_counts["user_compliments"] += 1

        for friend_row in extract_user_friends(raw):
            friends_writer.writerow(friend_row)
            user_counts["user_friends"] += 1
finally:
    for f in files_to_close:
        f.close()

user_counts


Counter({'user_friends': 6900124,
         'user_compliments': 1276015,
         'users': 1000000})

## 7. Трансформација на reviews

Во овој дел reviews се обработуваат со streaming пристап, односно ред по ред без вчитување на целата датотека во меморија, бидејќи `reviews` е една од најголемите датотеки во dataset-от.

Се задржуваат само редови со валидни клучеви, валиден timestamp и постоечки `user_id` и `business_id`, а резултатот се запишува во CSV датотека за табелата `reviews`.

In [8]:
def transform_review(row):
    return {
        "review_id": clean_text(row.get("review_id")),
        "user_id": clean_text(row.get("user_id")),
        "business_id": clean_text(row.get("business_id")),
        "stars": clean_float(row.get("stars")),
        "useful": clean_int(row.get("useful")),
        "funny": clean_int(row.get("funny")),
        "cool": clean_int(row.get("cool")),
        "text": clean_text(row.get("text")),
        "review_date": clean_datetime(row.get("date")),
    }

review_fields = ["review_id", "user_id", "business_id", "stars", "useful", "funny", "cool", "text", "review_date"]
review_counts = Counter()

with (CSV_DIR / "reviews.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=review_fields)
    writer.writeheader()

    for raw in iter_csv(FILES["review"]):
        row = transform_review(raw)
        if row["review_id"] is None or row["user_id"] is None or row["business_id"] is None or row["review_date"] is None:
            review_counts["dropped_missing_required"] += 1
            continue
        if row["business_id"] not in valid_business_ids or row["user_id"] not in valid_user_ids:
            review_counts["dropped_invalid_foreign_key"] += 1
            continue

        writer.writerow(row)
        review_counts["reviews"] += 1

review_counts


Counter({'reviews': 1999994})

## 8. Трансформација на tips

Во овој дел tips се обработуваат и се подготвуваат за внесување во PostgreSQL табелата `tips`.

Бидејќи tip податоците немаат оригинален `tip_id`, идентификаторот се генерира автоматски во PostgreSQL со `GENERATED ALWAYS AS IDENTITY`.

In [9]:
def transform_tip(row):
    return {
        "user_id": clean_text(row.get("user_id")),
        "business_id": clean_text(row.get("business_id")),
        "text": clean_text(row.get("text")),
        "tip_date": clean_datetime(row.get("date")),
        "compliment_count": clean_int(row.get("compliment_count")),
    }

tip_fields = ["user_id", "business_id", "text", "tip_date", "compliment_count"]
tip_counts = Counter()

with (CSV_DIR / "tips.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=tip_fields)
    writer.writeheader()

    for raw in iter_csv(FILES["tip"]):
        row = transform_tip(raw)
        if row["user_id"] is None or row["business_id"] is None or row["tip_date"] is None:
            tip_counts["dropped_missing_required"] += 1
            continue
        if row["business_id"] not in valid_business_ids or row["user_id"] not in valid_user_ids:
            tip_counts["dropped_invalid_foreign_key"] += 1
            continue

        writer.writerow(row)
        tip_counts["tips"] += 1

tip_counts


Counter({'tips': 754776})

## 9. Трансформација на checkins

Во processed `checkins.csv`, колоната `date` содржи array/list од timestamps за секој business. За PostgreSQL ова поле се нормализира во табелата `checkins`: еден ред по `business_id` и `checkin_time`.

In [10]:
checkin_fields = ["business_id", "checkin_time"]
checkin_counts = Counter()

with (CSV_DIR / "checkins.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=checkin_fields)
    writer.writeheader()

    for raw in iter_csv(FILES["checkin"]):
        business_id = clean_text(raw.get("business_id"))
        dates = parse_list(raw.get("date"))

        if business_id is None or not dates:
            checkin_counts["dropped_missing_required"] += 1
            continue
        if business_id not in valid_business_ids:
            checkin_counts["dropped_invalid_foreign_key"] += 1
            continue

        for part in dates:
            checkin_time = clean_datetime(part)
            if checkin_time is None:
                checkin_counts["dropped_invalid_timestamp"] += 1
                continue
            writer.writerow({"business_id": business_id, "checkin_time": checkin_time})
            checkin_counts["checkins"] += 1

checkin_counts


Counter({'checkins': 13356875})

## 10. Статистики после нормализацијата

In [11]:
metadata = {
    "dataset_scope": "processed_yelp_dataset",
    "source_dir": str(DATA_DIR),
    "valid_business_ids": len(valid_business_ids),
    "valid_user_ids": len(valid_user_ids),
    "business_counts": dict(business_counts),
    "user_id_counts": dict(user_id_counts),
    "user_counts": dict(user_counts),
    "review_counts": dict(review_counts),
    "tip_counts": dict(tip_counts),
    "checkin_counts": dict(checkin_counts),
    "notes": [
        "Processed users.csv contains elite summary columns, not exact elite year lists.",
    ],
}

(OUTPUT_DIR / "etl_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
metadata


{'dataset_scope': 'processed_yelp_dataset',
 'source_dir': 'processed',
 'valid_business_ids': 150346,
 'valid_user_ids': 1000000,
 'business_counts': {'businesses': 150346,
  'business_categories': 668549,
  'business_attributes': 1206820,
  'business_hours': 801015,
  'categories': 1311},
 'user_id_counts': {'users': 1000000},
 'user_counts': {'users': 1000000,
  'user_compliments': 1276015,
  'user_friends': 6900124},
 'review_counts': {'reviews': 1999994},
 'tip_counts': {'tips': 754776},
 'checkin_counts': {'checkins': 13356875},
 'notes': ['Processed users.csv contains elite summary columns, not exact elite year lists.']}

## 11. Релациски модел, schema и partitioning

По трансформацијата на processed Yelp CSV датотеките се добиваат следниве 11 PostgreSQL табели:

1. `businesses`
2. `categories`
3. `business_categories`
4. `business_attributes`
5. `business_hours`
6. `users`
7. `user_compliments`
8. `user_friends`
9. `reviews`
10. `tips`
11. `checkins`

Исто така, се прави партиционирање на табелите `reviews`, `tips` и `checkins` со `PARTITION BY RANGE` според timestamp колоната. За секоја од овие табели се креираат годишни партиции и default партиција за редови што не припаѓаат во ниту една од однапред дефинираните годишни партиции.


In [12]:
SCHEMA_SQL = r'''
DROP TABLE IF EXISTS checkins CASCADE;
DROP TABLE IF EXISTS tips CASCADE;
DROP TABLE IF EXISTS reviews CASCADE;
DROP TABLE IF EXISTS user_friends CASCADE;
DROP TABLE IF EXISTS user_compliments CASCADE;
DROP TABLE IF EXISTS business_hours CASCADE;
DROP TABLE IF EXISTS business_attributes CASCADE;
DROP TABLE IF EXISTS business_categories CASCADE;
DROP TABLE IF EXISTS categories CASCADE;
DROP TABLE IF EXISTS users CASCADE;
DROP TABLE IF EXISTS businesses CASCADE;

CREATE TABLE businesses (
    business_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    address TEXT,
    city TEXT,
    state TEXT,
    postal_code TEXT,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    stars NUMERIC(2,1),
    review_count INTEGER,
    is_open BOOLEAN
);

CREATE TABLE categories (
    category_name TEXT PRIMARY KEY
);

CREATE TABLE business_categories (
    business_id TEXT NOT NULL REFERENCES businesses(business_id),
    category_name TEXT NOT NULL REFERENCES categories(category_name),
    PRIMARY KEY (business_id, category_name)
);

CREATE TABLE business_attributes (
    business_id TEXT NOT NULL REFERENCES businesses(business_id),
    attribute_name TEXT NOT NULL,
    attribute_value JSONB,
    PRIMARY KEY (business_id, attribute_name)
);

CREATE TABLE business_hours (
    business_id TEXT NOT NULL REFERENCES businesses(business_id),
    day_of_week TEXT NOT NULL,
    open_time TIME,
    close_time TIME,
    PRIMARY KEY (business_id, day_of_week)
);

CREATE TABLE users (
    user_id TEXT PRIMARY KEY,
    name TEXT,
    review_count INTEGER,
    yelping_since TIMESTAMP,
    useful INTEGER,
    funny INTEGER,
    cool INTEGER,
    fans INTEGER,
    average_stars NUMERIC(3,2),
    elite_years_count INTEGER,
    elite_first_year INTEGER,
    elite_last_year INTEGER,
    is_elite BOOLEAN
);

CREATE TABLE user_compliments (
    user_id TEXT NOT NULL REFERENCES users(user_id),
    compliment_type TEXT NOT NULL,
    compliment_count INTEGER NOT NULL,
    PRIMARY KEY (user_id, compliment_type)
);

CREATE TABLE user_friends (
    user_id TEXT NOT NULL REFERENCES users(user_id),
    friend_user_id TEXT NOT NULL REFERENCES users(user_id),
    PRIMARY KEY (user_id, friend_user_id)
);

CREATE TABLE reviews (
    review_id TEXT NOT NULL,
    user_id TEXT NOT NULL REFERENCES users(user_id),
    business_id TEXT NOT NULL REFERENCES businesses(business_id),
    stars NUMERIC(2,1),
    useful INTEGER,
    funny INTEGER,
    cool INTEGER,
    text TEXT,
    review_date TIMESTAMP NOT NULL,
    PRIMARY KEY (review_id, review_date)
) PARTITION BY RANGE (review_date);

CREATE TABLE tips (
    tip_id BIGINT GENERATED ALWAYS AS IDENTITY,
    user_id TEXT NOT NULL REFERENCES users(user_id),
    business_id TEXT NOT NULL REFERENCES businesses(business_id),
    text TEXT,
    tip_date TIMESTAMP NOT NULL,
    compliment_count INTEGER,
    PRIMARY KEY (tip_id, tip_date)
) PARTITION BY RANGE (tip_date);

CREATE TABLE checkins (
    checkin_id BIGINT GENERATED ALWAYS AS IDENTITY,
    business_id TEXT NOT NULL REFERENCES businesses(business_id),
    checkin_time TIMESTAMP NOT NULL,
    PRIMARY KEY (checkin_id, checkin_time)
) PARTITION BY RANGE (checkin_time);

DO $$
DECLARE
    y INTEGER;
BEGIN
    FOR y IN 2004..2023 LOOP
        EXECUTE format('CREATE TABLE reviews_%s PARTITION OF reviews FOR VALUES FROM (%L) TO (%L)', y, y || '-01-01', (y + 1) || '-01-01');
        EXECUTE format('CREATE TABLE tips_%s PARTITION OF tips FOR VALUES FROM (%L) TO (%L)', y, y || '-01-01', (y + 1) || '-01-01');
        EXECUTE format('CREATE TABLE checkins_%s PARTITION OF checkins FOR VALUES FROM (%L) TO (%L)', y, y || '-01-01', (y + 1) || '-01-01');
    END LOOP;
END $$;

CREATE TABLE reviews_default PARTITION OF reviews DEFAULT;
CREATE TABLE tips_default PARTITION OF tips DEFAULT;
CREATE TABLE checkins_default PARTITION OF checkins DEFAULT;
'''

(SQL_DIR / "01_schema.sql").write_text(SCHEMA_SQL, encoding="utf-8")
print((SQL_DIR / "01_schema.sql").resolve())


/mnt/c/Users/RazorVision/Desktop/NBP_Proekt/postgres/postgres_output/sql/01_schema.sql


## 12. PostgreSQL COPY внесување

Во овој дел нормализираните CSV датотеки се внесуваат во PostgreSQL со командата `COPY`.

`COPY` се користи затоа што овозможува масовно внесување на редови од CSV датотека, што е побрзо од извршување поединечна `INSERT` наредба за секој ред.

Пример за `COPY` наредба:

```sql
COPY businesses
FROM STDIN
WITH (FORMAT csv, HEADER true, NULL '');

In [13]:
RUN_DATABASE_LOAD = True  # Set to False if you want to skip the database load step
DB_DSN = "dbname=nbp_project user=nbp password=nbp123 host=localhost port=5432"

COPY_COMMANDS = [
    ("businesses", "businesses.csv", None),
    ("categories", "categories.csv", None),
    ("business_categories", "business_categories.csv", None),
    ("business_attributes", "business_attributes.csv", None),
    ("business_hours", "business_hours.csv", None),
    ("users", "users.csv", None),
    ("user_compliments", "user_compliments.csv", None),
    ("user_friends", "user_friends.csv", None),
    ("reviews", "reviews.csv", None),
    ("tips", "tips.csv", "(user_id, business_id, text, tip_date, compliment_count)"),
    ("checkins", "checkins.csv", "(business_id, checkin_time)"),
]

if RUN_DATABASE_LOAD:
    import psycopg

    with psycopg.connect(DB_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute(SCHEMA_SQL)
            conn.commit()

            for table, filename, columns in COPY_COMMANDS:
                columns_sql = f" {columns}" if columns else ""
                copy_sql = f"COPY {table}{columns_sql} FROM STDIN WITH (FORMAT csv, HEADER true, NULL '')"
                path = CSV_DIR / filename
                print("COPY", table, path)
                with path.open("r", encoding="utf-8") as f:
                    with cur.copy(copy_sql) as copy:
                        while data := f.read(1024 * 1024):
                            copy.write(data)
                conn.commit()
            
else:
    print("Database load is disabled. Set RUN_DATABASE_LOAD = True if you want to run COPY from Python.")


COPY businesses postgres_output/csv/businesses.csv
COPY categories postgres_output/csv/categories.csv
COPY business_categories postgres_output/csv/business_categories.csv
COPY business_attributes postgres_output/csv/business_attributes.csv
COPY business_hours postgres_output/csv/business_hours.csv
COPY users postgres_output/csv/users.csv
COPY user_compliments postgres_output/csv/user_compliments.csv
COPY user_friends postgres_output/csv/user_friends.csv
COPY reviews postgres_output/csv/reviews.csv
COPY tips postgres_output/csv/tips.csv
COPY checkins postgres_output/csv/checkins.csv
